In [1]:
import os
os.listdir("/kaggle/input")


['datasets', 'playground-series-s6e2']

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [3]:
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

In [4]:
train["Heart Disease"] = train["Heart Disease"].map({"Absence": 0, "Presence": 1})

In [5]:
cat_cols = [
    'Sex', 'Chest pain type', 'FBS over 120', 'EKG results',
    'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium',
    'Age_bin', 'Chol_bin', 'ST_depr_severity'   # tambahkan ini
]

In [6]:
def add_features(df):
    df = df.copy()
    
    # fitur lain tetap sama
    df['Age_bin']         = pd.cut(df['Age'], bins=[0,40,50,60,70,100], labels=[0,1,2,3,4]).astype('int')
    df['Chol_bin']        = pd.cut(df['Cholesterol'], bins=[0,200,240,300,600], labels=[0,1,2,3]).astype('int')
    
    df['BP_risk']         = (df['BP'] > 140).astype(int) + (df['BP'] > 160).astype(int)
    df['HR_reserve']      = df['Max HR'] - df['Age'] * 0.85
    df['Chol_Age_ratio']  = df['Cholesterol'] / (df['Age'] + 1)
    
    # Solusi aman untuk ST_depr_severity
    # 1. isi NaN dulu
    st_dep = df['ST depression'].fillna(0)
    
    # 2. gunakan numpy.digitize (lebih sederhana, langsung integer, tahan NaN)
    import numpy as np
    
    bins = [0, 1, 2, 5]           # batas kiri terbuka, kanan tertutup
    labels = [0, 1, 2, 3]         # 0 = ≤0, 1 = (0–1], 2 = (1–2], 3 = >2
    
    df['ST_depr_severity'] = np.digitize(st_dep, bins) - 1   # -1 supaya mulai dari 0
    
    # pastikan tidak ada nilai negatif (safety net)
    df['ST_depr_severity'] = df['ST_depr_severity'].clip(lower=0)
    
    # fitur lain
    df['Vessels_risk']    = (df['Number of vessels fluro'] >= 2).astype(int)
    df['Thallium_high']   = (df['Thallium'] == 7).astype(int)
    
    return df

In [7]:
train = add_features(train)
test = add_features(test)

drop = ['id', 'Heart Disease']
X = train.drop(columns=drop)
y = train['Heart Disease']
X_test = test.drop(columns=['id'])

In [8]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.04,
    'num_leaves': 150,
    'max_depth': 10,
    'min_child_samples': 40,
    'subsample': 0.85,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.8,
    'reg_lambda': 1.2,
    'random_state': 42,
    'verbosity': -1
}

skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)  # naik ke 7 folds

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (trn, val) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}/7")
    X_tr, X_val = X.iloc[trn], X.iloc[val]
    y_tr, y_val = y.iloc[trn], y.iloc[val]
    
    trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=trn_data)
    
    model = lgb.train(
        params,
        trn_data,
        num_boost_round=6000,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(300)]
    )
    
    oof[val] = model.predict(X_val)
    test_preds += model.predict(X_test) / skf.n_splits

print("CV AUC:", roc_auc_score(y, oof))

Fold 1/7
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[351]	valid_0's auc: 0.955356
Fold 2/7
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[342]	valid_0's auc: 0.954829
Fold 3/7
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[318]	valid_0's auc: 0.954391
Fold 4/7
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[351]	valid_0's auc: 0.955068
Fold 5/7
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[438]	valid_0's auc: 0.954026
Fold 6/7
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[358]	valid_0's auc: 0.956182
Fold 7/7
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[305]	valid_0's auc: 0.95513
CV AUC: 0.9549915066950231


In [9]:
full_data = lgb.Dataset(X, label=y, categorical_feature=cat_cols)
final = lgb.train(params, full_data, num_boost_round=int(model.best_iteration * 1.1))

pred = final.predict(X_test)

sub = pd.DataFrame({'id': test['id'], 'Heart Disease': pred})
sub.to_csv('submission_lgbm_fe_v2.csv', index=False)